In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import glob
from astropy.timeseries import LombScargle

from wotan import flatten

In [ ]:
BASE_PATH = (
    "/home/luciam/Documents/starry/"
    "Orion_yceb/dat_orion_sergio/sample_3/"
    "831359(1)/Light_Curve_Files/"
)

ORBITAL_CATALOG = (
    "orbital_adopted_periods_v1.csv"
)
# ==================================================
# FIND FILES
# ==================================================

files = sorted(
    glob.glob(
        os.path.join(
            BASE_PATH,
            "*.txt"
        )
    )
)

print(
    f"Found {len(files)} files"
)

# ==================================================
# GROUP FILES BY TIC
# ==================================================

tic_files = {}

for f in files:

    fname = os.path.basename(f)

    match = re.search(
        r'tic\s+(\d+)_S_(\d+)_LC',
        fname
    )

    if match is None:
        continue

    tic = int(
        match.group(1)
    )

    sector = int(
        match.group(2)
    )

    tic_files.setdefault(
        tic,
        []
    )

    tic_files[tic].append(
        (sector, f)
    )

print(
    f"Found {len(tic_files)} TICs"
)

In [ ]:
orbital_df = pd.read_csv(
    ORBITAL_CATALOG
)

orbital_df

In [ ]:
def get_tic_info(tic):

    row = orbital_df[
        orbital_df["tic"] == tic
    ].iloc[0]

    P_orb = row["P_adopted"]

    sectors = [
        int(x)
        for x in row["keep_sectors"].split(";")
    ]

    return (
        P_orb,
        sectors
    )

In [ ]:
def load_sector_file(fname):

    df = pd.read_csv(
        fname,
        delim_whitespace=True,
        comment="#",
        header=None
    )

    arr = df.values

    time_col = None
    flux_col = None

    for i in range(arr.shape[1]):

        med = np.nanmedian(
            arr[:, i]
        )

        if (
            time_col is None
            and
            1000 < med < 5000
        ):
            time_col = i

        if (
            flux_col is None
            and
            0.5 < med < 1.5
        ):
            flux_col = i

    if time_col is None:
        raise ValueError(
            f"TIME column not found in {fname}"
        )

    if flux_col is None:
        raise ValueError(
            f"FLUX column not found in {fname}"
        )

    time = arr[:, time_col]
    flux = arr[:, flux_col]

    mask = (
        np.isfinite(time)
        &
        np.isfinite(flux)
    )

    time = time[mask]
    flux = flux[mask]

    srt = np.argsort(time)

    return (
        time[srt],
        flux[srt]
    )

In [ ]:
def mask_eclipses(
    time,
    flux,
    P_orb,
    width=0.10
):

    phase = (
        (time % P_orb)
        / P_orb
    )

    mask = np.ones(
        len(time),
        dtype=bool
    )

    primary = (
        np.abs(
            phase - 0.5
        )
        < width
    )

    secondary = (
        phase < width
    ) | (
        phase > 1-width
    )

    mask[
        primary | secondary
    ] = False

    return (
        time[mask],
        flux[mask]
    )

In [ ]:
def detrend_rotation(
    time,
    flux
):

    flat, trend = flatten(
        time,
        flux,
        method="biweight",
        window_length=1.5,
        return_trend=True
    )

    return flat

In [ ]:
def get_rotation_periods(
    time,
    flux,
    min_period=0.2,
    max_period=20
):

    frequency = np.linspace(
        1/max_period,
        1/min_period,
        50000
    )

    ls = LombScargle(
        time,
        flux
    )

    power = ls.power(
        frequency
    )

    period = 1/frequency

    idx = np.argsort(
        power
    )[::-1]

    top_periods = []
    top_powers = []

    for i in idx[:10]:

        top_periods.append(
            period[i]
        )

        top_powers.append(
            power[i]
        )

    return (
        period,
        power,
        top_periods,
        top_powers
    )

In [ ]:
def inspect_rotation_tic(
    tic
):

    P_orb, keep_sectors = (
        get_tic_info(tic)
    )

    nsec = len(
        keep_sectors
    )

    fig, axes = plt.subplots(
        nsec,
        3,
        figsize=(15,4*nsec)
    )

    if nsec == 1:
        axes = np.array([axes])

    adopted_candidates = []

    for i, sector in enumerate(
        keep_sectors
    ):

        fname = dict(
            tic_files[tic]
        )[sector]

        time, flux = (
            load_sector_file(
                fname
            )
        )

        flux = (
            flux /
            np.nanmedian(flux)
        )

        # -------------------
        # remove eclipses
        # -------------------

        time_masked, flux_masked = (
            mask_eclipses(
                time,
                flux,
                P_orb
            )
        )

        # -------------------
        # detrend
        # -------------------

        flux_flat = (
            detrend_rotation(
                time_masked,
                flux_masked
            )
        )

        # -------------------
        # LS
        # -------------------

        period, power, top_periods, top_powers = (
            get_rotation_periods(
                time_masked,
                flux_flat
            )
        )

        adopted_candidates.append(
            top_periods[0]
        )

        # -------------------
        # original LC
        # -------------------

        axes[i,0].scatter(
            time,
            flux,
            s=2
        )

        axes[i,0].set_title(
            f"Sector {sector}"
        )

        # -------------------
        # masked LC
        # -------------------

        axes[i,1].scatter(
            time_masked,
            flux_flat,
            s=2
        )

        axes[i,1].set_title(
            "Masked + Detrended"
        )

        # -------------------
        # periodogram
        # -------------------

        axes[i,2].plot(
            period,
            power
        )

        axes[i,2].axvline(
            top_periods[0],
            ls="--"
        )

        axes[i,2].set_title(
            f"P={top_periods[0]:.3f} d"
        )

        print(
            f"\nSector {sector}"
        )

        for p, pw in zip(
            top_periods[:5],
            top_powers[:5]
        ):

            print(
                f"{p:.5f}",
                pw
            )

    plt.suptitle(
        f"TIC {tic}\n"
        f"P_orb={P_orb:.4f} d"
    )

    plt.tight_layout()

    plt.show()

    return adopted_candidates

In [ ]:
inspect_rotation_tic(
    11300044
)

In [ ]:
P_orb = 2.101805

sector = 98

fname = dict(tic_files[11300044])[sector]

time, flux = load_sector_file(fname)

flux = flux / np.nanmedian(flux)

phase = (time % P_orb) / P_orb

plt.figure(figsize=(8,4))

plt.scatter(
    phase,
    flux,
    s=1,
    alpha=0.2
)

plt.xlabel("Phase")
plt.ylabel("Flux")

plt.show()

In [ ]:
def find_eclipse_phases(
    time,
    flux,
    P_orb,
    nbins=120
):

    phase = (
        (time % P_orb)
        / P_orb
    )

    bins = np.linspace(
        0,
        1,
        nbins+1
    )

    centers = 0.5*(
        bins[:-1] +
        bins[1:]
    )

    medians = np.full(
        nbins,
        np.nan
    )

    for i in range(nbins):

        m = (
            (phase >= bins[i]) &
            (phase < bins[i+1])
        )

        if np.sum(m) > 5:

            medians[i] = np.nanmedian(
                flux[m]
            )

    idx = np.argsort(
        medians
    )

    primary = centers[
        idx[0]
    ]

    secondary = centers[
        idx[1]
    ]

    return (
        primary,
        secondary
    )

In [ ]:
p1, p2 = find_eclipse_phases(
    time,
    flux,
    P_orb
)

print(
    p1,
    p2
)

In [ ]:
def find_eclipse_phases(
    time,
    flux,
    P_orb,
    nbins=120
):

    phase = (
        (time % P_orb)
        / P_orb
    )

    bins = np.linspace(
        0,
        1,
        nbins + 1
    )

    centers = 0.5 * (
        bins[:-1] +
        bins[1:]
    )

    medians = np.full(
        nbins,
        np.nan
    )

    for i in range(nbins):

        m = (
            (phase >= bins[i]) &
            (phase < bins[i+1])
        )

        if np.sum(m) > 5:

            medians[i] = np.nanmedian(
                flux[m]
            )

    # mínimo principal

    idx1 = np.nanargmin(
        medians
    )

    phase1 = centers[
        idx1
    ]

    # buscar segundo eclipse lejos del primero

    distance = np.abs(
        centers - phase1
    )

    distance = np.minimum(
        distance,
        1 - distance
    )

    valid = (
        distance > 0.20
    )

    idx2 = np.where(
        valid
    )[0]

    idx2 = idx2[
        np.nanargmin(
            medians[idx2]
        )
    ]

    phase2 = centers[
        idx2
    ]

    return (
        phase1,
        phase2,
        centers,
        medians
    )

In [ ]:
tic = 11300044

P_orb = 2.101805

sector = 98

fname = dict(
    tic_files[tic]
)[sector]

time, flux = load_sector_file(
    fname
)

flux = (
    flux /
    np.nanmedian(flux)
)

p1, p2, centers, medians = (
    find_eclipse_phases(
        time,
        flux,
        P_orb
    )
)

print(
    "eclipse 1 =",
    p1
)

print(
    "eclipse 2 =",
    p2
)

In [ ]:
phase = (
    (time % P_orb)
    / P_orb
)

plt.figure(
    figsize=(8,4)
)

plt.scatter(
    phase,
    flux,
    s=1,
    alpha=0.1
)

plt.axvline(
    p1,
    color="red"
)

plt.axvline(
    p2,
    color="orange"
)

plt.xlabel(
    "Phase"
)

plt.ylabel(
    "Flux"
)

plt.show()

In [ ]:
def mask_eclipses(
    time,
    flux,
    P_orb,
    width=0.06
):

    p1, p2, _, _ = find_eclipse_phases(
        time,
        flux,
        P_orb
    )

    phase = (
        (time % P_orb)
        / P_orb
    )

    d1 = np.abs(
        phase - p1
    )

    d2 = np.abs(
        phase - p2
    )

    mask = (
        (d1 > width) &
        (d2 > width)
    )

    return (
        time[mask],
        flux[mask]
    )

In [ ]:
time_masked, flux_masked = mask_eclipses(
    time,
    flux,
    P_orb,
    width=0.06
)

print(
    len(time),
    len(time_masked)
)

In [ ]:
phase = (
    (time % P_orb)
    / P_orb
)

phase_masked = (
    (time_masked % P_orb)
    / P_orb
)

plt.figure(figsize=(10,4))

plt.scatter(
    phase,
    flux,
    s=1,
    alpha=0.05,
    label="original"
)

plt.scatter(
    phase_masked,
    flux_masked,
    s=1,
    alpha=0.3,
    label="masked"
)

plt.xlabel("Phase")
plt.ylabel("Flux")

plt.legend()

plt.show()